<a href="https://colab.research.google.com/github/harshvardhaniimi/isa383-google-colab/blob/main/10_Pandas_Advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas for Data Wrangling (Advanced) 🚢

### ISA 383: Python for Business Analytics

This is the second of two notebooks on pandas.
The Basic notebook taught you to read, inspect, reshape, select, filter, and sort a single table.
pandas has many advanced tools, and you do not need all of them to do good analysis.
This notebook focuses on a few tools that answer most business questions: `merge()` to **combine** tables, and `groupby()` with `agg()` and `transform()` to **summarize** them.
Section 5 gives a short tour of several other tools, with links to their documentation for when you want to learn more.
We continue the investigation of the RMS *Titanic* passenger list.

## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.

## Learning objectives 🎯

By the end of this notebook, you should be able to:

1. Use selection and filtering to answer a quick question about the passenger list.
2. Combine two related tables with `merge()`, choose the correct join type, and check the result with `validate=` and `indicator=True`.
3. Summarize groups with `groupby()`, using one or two grouping keys.
4. Build readable summary tables with named aggregation in `agg()`, and add a group-level value to every row with `transform()`.
5. Recognize when `pivot_table()`, `crosstab()`, `cut()`, and `concat()` are useful, and know where to read more about them.

## Where we left off 🧊

In April 1912, the *Titanic* sank on its maiden voyage after striking an iceberg, with lifeboat space for only about half of the 2,224 people aboard.
Two official inquiries asked who had been saved and why.
In the Basic notebook, our filters showed that 38.4% of the 891 passengers in our file survived, that women survived far more often than men, and that first-class passengers survived far more often than third-class passengers.

Those comparisons looked at one variable at a time.
Real questions usually involve several variables together, and the information we need is often spread over several tables.
Your remaining inquiry questions:

| # | Inquiry question | pandas skill |
|---|---|---|
| Q6 | Did first class buy a better chance, and did that depend on sex? | `groupby()` with two keys |
| Q7 | Did the port of embarkation matter, or was it really ticket class? | `merge()`, `groupby()`, `crosstab()` |
| Q8 | How much did each passenger really pay when several people shared a ticket? | `groupby()` + `transform()` |

Look for the 🕵️ symbol in the cells that answer them.

## Setup 🧰

We download the course data from the course GitHub repository, load the passenger file and the `ports` sheet from the Excel workbook, and recreate the `family_size` column from the Basic notebook.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

from urllib.request import urlretrieve

if Path("/content").exists():
    DATA_DIR = Path("/content/isa383/data")
else:
    DATA_DIR = Path.home() / "isa383" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download the course data from the course GitHub repository (only the first time).
DATA_URL = "https://raw.githubusercontent.com/harshvardhaniimi/isa383-google-colab/main/data/"
for filename in ["titanic.csv", "titanic_extra.xlsx"]:
    if not (DATA_DIR / filename).exists():
        urlretrieve(DATA_URL + filename, DATA_DIR / filename)
print("Course data is ready.")

titanic = pd.read_csv(DATA_DIR / "titanic.csv")
ports = pd.read_excel(DATA_DIR / "titanic_extra.xlsx", sheet_name="ports")

titanic["family_size"] = titanic["SibSp"] + titanic["Parch"] + 1

print("titanic:", titanic.shape, "| ports:", ports.shape)

Course data is ready.
titanic: (891, 13) | ports: (4, 6)


In [2]:
titanic.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


## 1. Warm-up: selection and filtering 🔎

Every analysis in this notebook builds on two skills from the Basic notebook: choosing the columns you need, and keeping only the rows that match a condition.
Here is a short reminder of the patterns.

```python
df[["column_a", "column_b"]]                 # select columns
df[(df["a"] == x) & (df["b"] < y)]           # filter rows; wrap each condition in parentheses
df.loc[mask, ["column_a", "column_b"]]       # filter rows AND select columns
```

Documentation: [Boolean indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)

In [3]:
# Survival of women in first class and in third class, using filters only.
women_first = titanic[(titanic["Sex"] == "female") & (titanic["Pclass"] == 1)]
women_first = titanic.query("Sex == 'female' and Pclass == 1")

women_third = titanic[(titanic["Sex"] == "female") & (titanic["Pclass"] == 3)]
women_third = titanic.query("Sex == 'female' and Pclass == 3")

print(f"First-class women: {women_first['Survived'].mean():.1%} survived (n = {len(women_first)})")
print(f"Third-class women: {women_third['Survived'].mean():.1%} survived (n = {len(women_third)})")

First-class women: 96.8% survived (n = 94)
Third-class women: 50.0% survived (n = 144)


In [6]:
titanic.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin',
       'Embarked', 'family_size'],
      dtype='object')

In [8]:
titanic["Fare"].mean(), titanic["Fare"].max(), titanic["Fare"].min()

(np.float64(32.204207968574636), 512.3292, 0.0)

In [10]:
fare_threshold = 100
titanic.loc[titanic["Fare"]>fare_threshold].head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.0000,C23 C25 C27,S,6
31,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C,2
88,89,1,1,"Fortune, Miss. Mabel Helen",female,23.0,3,2,19950,263.0000,C23 C25 C27,S,6
118,119,0,1,"Baxter, Mr. Quigg Edmond",male,24.0,0,1,PC 17558,247.5208,B58 B60,C,2
195,196,1,1,"Lurette, Miss. Elise",female,58.0,0,0,PC 17569,146.5208,B80,C,1


In [13]:
titanic.query("Fare > @fare_threshold").head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.0000,C23 C25 C27,S,6
31,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C,2
88,89,1,1,"Fortune, Miss. Mabel Helen",female,23.0,3,2,19950,263.0000,C23 C25 C27,S,6
118,119,0,1,"Baxter, Mr. Quigg Edmond",male,24.0,0,1,PC 17558,247.5208,B58 B60,C,2
195,196,1,1,"Lurette, Miss. Elise",female,58.0,0,0,PC 17569,146.5208,B80,C,1


Two filters gave us two numbers.
To compare every combination of class and sex in this way, we would need six filters and six lines of nearly identical code.
`groupby()` in Section 3 gives all six numbers in one line.

### 🧩 Guided practice: A quick filter

Use `loc` with a filter to show the `Name`, `Age`, and `Fare` of the first-class passengers who boarded at Queenstown (`Embarked == "Q"`).
How many are there, and what does that suggest about who boarded in Ireland?

In [15]:
# TODO: Filter first-class passengers from Queenstown and select three columns.
titanic.query("Embarked == 'Q' and Pclass == 1")[['Name', 'Age', 'Fare']]

,Name,Age,Fare
245,"Minahan, Dr. William Edward",44.0,90.0
412,"Minahan, Miss. Daisy E",33.0,90.0


## 2. Combining related tables with `merge()` 🔗

`merge()` matches rows from two tables using the values in one or more **key** columns, like a SQL `JOIN` or Excel's `XLOOKUP`.
Before merging, ask two questions.

**What is the relationship between the keys?**
Many passengers boarded at one port, so passengers-to-ports is **many-to-one**.
One-to-one and many-to-many relationships also exist; many-to-many is usually a warning sign.

**Which rows should survive the merge?**

| `how=` | Keeps | Typical use |
|---|---|---|
| `"inner"` | only keys found in **both** tables | analysis where unmatched rows are useless |
| `"left"` | every row of the **left** table | enriching a main table with a lookup (the most common case) |
| `"right"` | every row of the **right** table | rarely needed; swap the tables and use `"left"` |
| `"outer"` | every row from **both** tables | auditing: finding what did not match |

### 🧭 Syntax first: Merging tables

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
left.merge(
    right,
    how="left",                         # "inner", "left", "right", or "outer"
    on="key",                           # when the key has the same name in both tables
    left_on="key_a", right_on="key_b",  # when the names differ
    suffixes=("_x", "_y"),              # added to other columns that share a name
    validate="many_to_one",             # raise an error if the relationship is wrong
    indicator=True,                     # add a _merge column: both / left_only / right_only
)
```

Documentation: [DataFrame.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) · [Merge guide](https://pandas.pydata.org/docs/user_guide/merging.html)

In [16]:
# The lookup table: one row per port the ship left from.
ports

,code,port,country_today,departure_date,stop_number,note
0,B,Belfast,United Kingdom,1912-04-02,0,"Shipyard; delivery trip only, no passengers in..."
1,S,Southampton,United Kingdom,1912-04-10,1,Maiden voyage begins
2,C,Cherbourg,France,1912-04-10,2,Evening call in France
3,Q,Queenstown,Ireland,1912-04-11,3,Last stop before the Atlantic crossing (now ca...


In [20]:
with_ports = titanic.merge(
    ports[["code", "port", "departure_date", "stop_number"]],
    how="left",
    left_on="Embarked",      # the key is called Embarked in titanic ...
    right_on="code",         # ... and code in ports
    # validate="many_to_one",  # each port code must appear only once in ports
    indicator=True,
)

with_ports[["Name", "Embarked", "code", "port", "departure_date", "_merge"]].head()

,Name,Embarked,code,port,departure_date,_merge
0,"Braund, Mr. Owen Harris",S,S,Southampton,1912-04-10,both
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,C,Cherbourg,1912-04-10,both
2,"Heikkinen, Miss. Laina",S,S,Southampton,1912-04-10,both
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,S,Southampton,1912-04-10,both
4,"Allen, Mr. William Henry",S,S,Southampton,1912-04-10,both


In [21]:
with_ports["_merge"].value_counts()

,count
_merge,
both,889
left_only,2
right_only,0


In [22]:
# Which passengers found no matching port?
with_ports.loc[with_ports["_merge"] == "left_only", ["Name", "Ticket", "Cabin", "Embarked", "port"]]

,Name,Ticket,Cabin,Embarked,port
61,"Icard, Miss. Amelie",113572,B28,NaN,NaN
829,"Stone, Mrs. George Nelson (Martha Evelyn)",113572,B28,NaN,NaN


Miss Amelie Icard and Mrs. George Nelson Stone shared ticket 113572 and cabin B28, but their port of embarkation is missing.
A left merge kept them, with `NaN` for every port column, and `indicator=True` made them easy to find.
*Encyclopedia Titanica* records that both boarded at Southampton; we will use that kind of evidence when we fill missing values in the cleaning notebook.

In [23]:
# How many rows does each join type keep? A loop makes the comparison easy.
for how in ["inner", "left", "right", "outer"]:
    result = titanic.merge(ports, how=how, left_on="Embarked", right_on="code")
    print(f"{how:>5}: {len(result):>4} rows")

inner:  889 rows
 left:  891 rows
right:  890 rows
outer:  892 rows


- `inner` dropped the two passengers with no port (889 rows).
- `left` kept all 891 passengers.
- `right` kept every port, so Belfast appears once with no passenger attached (889 + 1 = 890 rows).
- `outer` kept everything from both sides (892 rows).

⚠️ If a key value appears several times in **both** tables (many-to-many), `merge()` pairs every match with every other match.
That is the most common way an analysis silently gains extra rows.
The `validate=` argument raises an error when the relationship is not the one you expect, so use it whenever you merge.

> 💭 **Quick thought question:** After a left merge, the result has more rows than the left table.
What does that tell you about the key in the right table?

### 🧩 Guided practice: Inner merge

Merge `titanic` with `ports` using `how="inner"`.
How many rows does the result have?
Use `value_counts()` on the `port` column to count passengers per port, and explain in one sentence why two passengers are missing.

In [25]:
# TODO: Inner merge, then count passengers per port.
df2 = titanic.merge(ports, left_on="Embarked", right_on = "code", how = "inner")
print(df2.shape)

df2["port"].value_counts()

(889, 19)


,count
port,
Southampton,644
Cherbourg,168
Queenstown,77


## 3. `groupby()`: split, apply, combine 🧮

Most business questions compare groups: revenue by region, churn by plan, survival by class.
`groupby()` follows a three-step pattern known as **split-apply-combine**.

```text
                 split by Pclass             apply mean()          combine
891 rows  ──┬──> Pclass 1 (216 rows) ──────> 0.630 ──┐
            ├──> Pclass 2 (184 rows) ──────> 0.473 ──┼──> one row per class
            └──> Pclass 3 (491 rows) ──────> 0.242 ──┘
```

The result has one row per group.
With two grouping keys, the result has one row per *combination* of keys.

### 🧭 Syntax first: Grouping

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.groupby("key")["value"].mean()
df.groupby(["key_a", "key_b"])["value"].mean()
df.groupby("key", as_index=False)["value"].mean()   # keys stay as ordinary columns

df.groupby("key").size()             # rows per group
df.groupby("key")["value"].count()   # NON-MISSING values per group
series_with_two_keys.unstack()       # move the inner key into the columns
```

Documentation: [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [GroupBy guide](https://pandas.pydata.org/docs/user_guide/groupby.html) · [unstack](https://pandas.pydata.org/docs/reference/api/pandas.Series.unstack.html)

In [29]:
# 🕵️ Q6 (part 1): Did first class buy a better chance?
df = titanic.groupby("Pclass", as_index = False)["Survived"].mean()
df

,Pclass,Survived
0,1,0.629630
1,2,0.472826
2,3,0.242363


In [30]:
df.columns

Index(['Pclass', 'Survived'], dtype='object')

In [31]:
# size() counts rows; count() counts non-missing values in one column.
by_class = titanic.groupby("Pclass")

pd.DataFrame(
    {
        "passengers": by_class.size(),
        "ages_recorded": by_class["Age"].count(),
        "survivors": by_class["Survived"].sum(),
    }
)

,passengers,ages_recorded,survivors
Pclass,,,
1,216,186,136
2,184,173,87
3,491,355,119


In [32]:
# Two keys: one row per (class, sex) combination.
rate_by_class_sex = titanic.groupby(["Pclass", "Sex"])["Survived"].mean().round(3)
rate_by_class_sex

Pclass  Sex   
1       female    0.968
        male      0.369
2       female    0.921
        male      0.157
3       female    0.500
        male      0.135
Name: Survived, dtype: float64

In [33]:
# unstack() turns the long result into a wide table that is easier to read.
rate_by_class_sex.unstack()

Sex,female,male
Pclass,,
1,0.968,0.369
2,0.921,0.157
3,0.500,0.135


This small table carries the main story of the disaster.
Within every class, women survived far more often than men.
Within each sex, survival fell with class: 96.8% of first-class women survived, but only 50.0% of third-class women; 36.9% of first-class men survived, but only 13.5% of third-class men.
Sex mattered most, and class mattered within each sex.

In [34]:
# 🕵️ Q7 (part 1): Did the port of embarkation matter?
with_ports.groupby("port")["Survived"].mean().round(3).sort_values(ascending=False)

,Survived
port,
Cherbourg,0.554
Queenstown,0.390
Southampton,0.337


Passengers who boarded at Cherbourg survived far more often than those from Southampton.
Did boarding in France protect people, or were different kinds of people boarding there?
We return to this in Section 5.

> 💭 **Quick thought question:** Why is `titanic.groupby("Pclass").size()` different from `titanic.groupby("Pclass")["Age"].count()`?
Which one should you use as the denominator of a survival rate?

### 🧩 Guided practice: Sex and port

Compute the survival rate for every combination of `Sex` and `Embarked`, then `unstack()` the result so that ports are columns.
Does the Cherbourg advantage hold for both women and men?

In [36]:
# TODO: Group by Sex and Embarked, compute the mean of Survived, then unstack.
titanic.groupby(['Sex', 'Pclass', 'Embarked'])['Survived'].mean()

Sex     Pclass  Embarked
female  1       C           0.976744
                Q           1.000000
                S           0.958333
        2       C           1.000000
                Q           1.000000
                S           0.910448
        3       C           0.652174
                Q           0.727273
                S           0.375000
male    1       C           0.404762
                Q           0.000000
                S           0.354430
        2       C           0.200000
                Q           0.000000
                S           0.154639
        3       C           0.232558
                Q           0.076923
                S           0.128302
Name: Survived, dtype: float64

## 4. Summaries with `agg()` and `transform()` 📊

One number per group is often not enough.
`agg()` computes several summaries at once, and **named aggregation** gives each output column a readable name.
`transform()` is different: it returns one value **per original row**, so you can compare each passenger with their group.

### 🧭 Syntax first: Aggregating and transforming groups

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.groupby("key")["value"].agg(["mean", "median", "max"])

df.groupby("key").agg(
    output_name=("column", "function"),          # built-in name such as "mean"
    other_output=("column", custom_function),    # or your own function
)

df.groupby("key")["value"].transform("mean")     # same length as df
```

Documentation: [agg](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html) · [Named aggregation](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation) · [transform](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html)

In [37]:
# A list of functions applied to one column.
titanic.groupby("Pclass")["Fare"].agg(["mean", "median", "min", "max"]).round(2)

,mean,median,min,max
Pclass,,,,
1,84.15,60.29,0.0,512.33
2,20.66,14.25,0.0,73.50
3,13.68,8.05,0.0,69.55


In every class the mean fare is well above the median, a sign of a few very expensive tickets pulling the average up (a right-skewed distribution).
For skewed money values, the median is usually the more honest "typical" value.

In [38]:
# Named aggregation: output_name=(input_column, function). This is the most readable form.
class_report = (
    titanic.groupby("Pclass")
    .agg(
        passengers=("PassengerId", "size"),
        survivors=("Survived", "sum"),
        survival_rate=("Survived", "mean"),
        median_age=("Age", "median"),
        median_fare=("Fare", "median"),
        women=("Sex", lambda values: (values == "female").sum()),
    )
    .round(3)
)
class_report

,passengers,survivors,survival_rate,median_age,median_fare,women
Pclass,,,,,,
1,216,136,0.630,37.0,60.287,94
2,184,87,0.473,29.0,14.250,76
3,491,119,0.242,24.0,8.050,144


In [39]:
# 🕵️ Q8: How much did each passenger really pay?
# transform("size") gives every row the number of passengers on its ticket.
titanic["ticket_group_size"] = titanic.groupby("Ticket")["PassengerId"].transform("size")
titanic["fare_per_person"] = (titanic["Fare"] / titanic["ticket_group_size"]).round(2)

titanic.loc[
    titanic["Ticket"].isin(["PC 17755", "CA. 2343"]),
    ["Name", "Ticket", "Fare", "ticket_group_size", "fare_per_person"],
]

,Name,Ticket,Fare,ticket_group_size,fare_per_person
159,"Sage, Master. Thomas Henry",CA. 2343,69.5500,7,9.94
180,"Sage, Miss. Constance Gladys",CA. 2343,69.5500,7,9.94
201,"Sage, Mr. Frederick",CA. 2343,69.5500,7,9.94
258,"Ward, Miss. Anna",PC 17755,512.3292,3,170.78
324,"Sage, Mr. George John Jr",CA. 2343,69.5500,7,9.94
679,"Cardeza, Mr. Thomas Drake Martinez",PC 17755,512.3292,3,170.78
737,"Lesurer, Mr. Gustave J",PC 17755,512.3292,3,170.78
792,"Sage, Miss. Stella Anna",CA. 2343,69.5500,7,9.94
846,"Sage, Mr. Douglas Bullen",CA. 2343,69.5500,7,9.94
863,"Sage, Miss. Dorothy Edith ""Dolly""",CA. 2343,69.5500,7,9.94


The Cardeza party's £512.33 works out to £170.78 per person among the three people in our file, and the Sage family's £69.55 ticket to £9.94 each.
Our file is only a subset, though: Charlotte Cardeza and four more members of the Sage family are missing from it.
Because `ticket_group_size` can only count companions who appear in our data, `fare_per_person` is an **upper bound** on what each person really paid.

In [ ]:
titanic.groupby("Pclass")[["Fare", "fare_per_person"]].median().round(2)

,Fare,fare_per_person
Pclass,,
1,60.29,35.25
2,14.25,13.00
3,8.05,7.85


> 💭 **Quick thought question:** `agg()` returned one row per class, while `transform()` returned one value for each of the 891 passengers.
Which would you use to add a "class average fare" column to the passenger table, and which to build a summary slide?

### 🧩 Guided practice: A port report

Use named aggregation to build a table with one row per `Embarked` value and these columns: `passengers`, `survival_rate`, `median_fare_per_person`, and `share_first_class`.
*Hint:* for the last one, use `("Pclass", lambda values: (values == 1).mean())`.

In [ ]:
# TODO: Group by Embarked and use named aggregation.

## 5. A short tour of other useful tools 🧭

pandas has many more tools for combining and summarizing data.
We look at a few of them briefly here: each one gets a short example, and the table links to its documentation.
Read the documentation when you need one of these tools in your own work.

| Tool | What it does | Documentation |
|---|---|---|
| `pivot_table()` | `groupby()` and `unstack()` in one call, with optional totals. Unlike `pivot()`, it aggregates duplicate rows. | [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) |
| `pd.crosstab()` | Counts how often combinations of two categories occur; `normalize=` turns the counts into shares. | [crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) |
| `pd.cut()`, `pd.qcut()` | Turn a numeric column into groups, with bin edges you choose (`cut`) or groups of equal size (`qcut`). | [cut](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) · [qcut](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html) |
| `pd.concat()` | Stacks tables that have the same columns, such as twelve monthly sales files, into one table. | [concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) |
| `df.join()` | A shortcut for `merge()` when the lookup table uses the key as its index. | [join](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) |
| `GroupBy.filter()` | Keeps or drops whole groups by a rule, such as "tickets shared by four or more people". | [filter](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.filter.html) |
| Method chaining | Writes several steps as one pipeline that reads from top to bottom, often with `assign()`. | [assign](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html) · [pipe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html) |

For a longer introduction to all of these, see the pandas user guides on [group by](https://pandas.pydata.org/docs/user_guide/groupby.html), [merging and concatenating](https://pandas.pydata.org/docs/user_guide/merging.html), and [reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html).

In [ ]:
# pivot_table(): the class x sex table from Section 3 in one call, with totals in the "All" row and column.
titanic.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean", margins=True).round(3)

Sex,female,male,All
Pclass,,,
1,0.968,0.369,0.630
2,0.921,0.157,0.473
3,0.500,0.135,0.242
All,0.742,0.189,0.384


In [ ]:
# cut(): bin ages into labeled groups. right=False makes each bin include its left edge: [0, 13), [13, 18), ...
titanic["age_group"] = pd.cut(
    titanic["Age"],
    bins=[0, 13, 18, 40, 60, 81],
    labels=["child (0-12)", "teen (13-17)", "adult (18-39)", "middle age (40-59)", "senior (60+)"],
    right=False,
)
titanic["age_group"].value_counts(sort=False, dropna=False)

age_group
child (0-12)           69
teen (13-17)           44
adult (18-39)         438
middle age (40-59)    137
senior (60+)           26
NaN                   177
Name: count, dtype: int64

In [ ]:
# 🕵️ Q7 (part 2): Was Cherbourg special, or did it carry different passengers?
# crosstab() with normalize="index" turns each row into shares that sum to 1.
pd.crosstab(with_ports["port"], with_ports["Pclass"], normalize="index").round(3)

Pclass,1,2,3
port,,,
Cherbourg,0.506,0.101,0.393
Queenstown,0.026,0.039,0.935
Southampton,0.197,0.255,0.548


Half of the Cherbourg passengers travelled first class, compared with about one in five from Southampton.
So much of the Cherbourg "advantage" reflects *who* boarded there rather than the port itself.
This is **confounding**: a third variable (here, class) is related to both the grouping (port) and the outcome (survival).

In [ ]:
# concat(): suppose the passenger list arrived as three files, one per class. Stack them back into one table.
files = [titanic[titanic["Pclass"] == ticket_class] for ticket_class in [1, 2, 3]]
print(files)
manifest = pd.concat(files, ignore_index=True)

print("Rows per file:", [len(f) for f in files], "-> rows after concat:", len(manifest))

[     PassengerId  Survived  Pclass                                               Name     Sex   Age  SibSp  Parch  \
1              2         1       1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1      0   
3              4         1       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1      0   
6              7         0       1                            McCarthy, Mr. Timothy J    male  54.0      0      0   
11            12         1       1                           Bonnell, Miss. Elizabeth  female  58.0      0      0   
23            24         1       1                       Sloper, Mr. William Thompson    male  28.0      0      0   
..           ...       ...     ...                                                ...     ...   ...    ...    ...   
871          872         1       1   Beckwith, Mrs. Richard Leonard (Sallie Monypeny)  female  47.0      1      1   
872          873         0       1                           Ca

## 6. Bringing it together: the inquiry's report 📝

**Q6: Class and sex.**
Survival depended on sex first and class second: 96.8% of first-class women survived, compared with 13.5% of third-class men.
Within every class, women survived far more often than men.

**Q7: Port.**
Cherbourg passengers survived more often (55.4%) mainly because half of them travelled first class, compared with about one in five from Southampton.

**Q8: Fares.**
Fares belong to tickets, and many tickets covered several people.
Dividing by the number of people on each ticket lowers the median first-class fare from £60.29 to at most £35.25 per person.

**Limitations.**
These patterns describe 891 of roughly 1,300 passengers and no crew; several groups are small; 177 ages are missing; and grouped comparisons show associations, not causes.

## Exercises 📝

For each exercise, show a compact result and write two or three sentences of interpretation, including at least one limitation.
Exercise 3 is a challenge exercise.

### Exercise 1: A class report for the inquiry

Use named aggregation to build one row per `Pclass` with: number of passengers, survivors, survival rate, median age, median `fare_per_person`, and share with a recorded cabin.
Then use `merge()` with the `class_names` table below so that the rows say "First", "Second", and "Third", and use `validate=` to check the relationship.
Sort the table by survival rate and write a three-sentence summary.

*Hint:* `groupby("Pclass", as_index=False)` keeps `Pclass` as a column, which makes the merge easier.

In [ ]:
# A small lookup table with readable class names.
class_names = pd.DataFrame({"Pclass": [1, 2, 3], "class_name": ["First", "Second", "Third"]})

# Write your solution here.

### Exercise 2: Did every child get the same chance?

Use the `age_group` column from Section 5.
Compute the survival rate for every combination of `age_group` and `Pclass`, with age groups as rows and classes as columns.
You can use `groupby()` with `unstack()`, or `pivot_table()`.
Build a second table with the number of passengers behind each rate.
Which group was left behind, and which rates are based on fewer than ten people?

In [ ]:
# Write your solution here.

### Exercise 3 (challenge): A reusable report function

Write a function `survival_report(df, by, min_size=1)` that returns, for any grouping column or list of columns `by`, the number of passengers, number of survivors, and survival rate, sorted from the highest rate to the lowest.
Groups with fewer than `min_size` passengers should be dropped (*hint:* a Boolean mask on the result, or `GroupBy.filter()` from the tour).
Test it with `"Sex"`, `["Pclass", "Sex"]`, `"age_group"`, and `"Embarked"`, using `min_size=10` at least once.

In [ ]:
# Write your solution here.


def survival_report(df, by, min_size=1):
    pass

## Summary ✅

- Selection and filtering answer one question at a time; `groupby()` answers the same question for every group at once.
- `merge()` matches rows on key columns. Choose `how=` deliberately, and use `validate=` and `indicator=True` to catch surprises.
- `groupby()` follows split-apply-combine. `size()` counts rows, `count()` counts non-missing values, and `unstack()` turns a two-key result into a readable table.
- Named aggregation in `agg()` produces readable summary tables; `transform()` returns one value per row for within-group comparisons.
- `pivot_table()`, `crosstab()`, `cut()`, `concat()`, `join()`, and `filter()` are worth knowing; their documentation is linked in Section 5. Always check group sizes, and look for confounding before you draw conclusions.

**Next:** *Data Manipulation, Cleaning and Imputation* deals with the missing ages, cabins, and ports we kept running into, along with outliers and type conversions.